# Project Title:
Monitoring Trends and Evaluating Intervention Impact of Lassa Fever in Nigeria.

*How can public health programs monitor disease trends and evaluate the impact of interventions using routine surveillance data?*

In [1]:
import pandas as pd
import numpy as np

np.random.seed(42)

# -----------------------------
# Configuration
# -----------------------------
states = ['Edo', 'Ondo', 'Ebonyi', 'Bauchi', 'Plateau', 'Taraba']
dates = pd.date_range(start='2019-01-01', end='2023-12-01', freq='MS')

intervention_start = pd.Timestamp('2021-01-01')

data = []

# -----------------------------
# Generate Data
# -----------------------------
for state in states:
    base_cases = np.random.randint(20, 45)

    for date in dates:
        # Seasonal effect (higher early year)
        month_factor = 1.3 if date.month in [1, 2, 3] else 0.8

        # Intervention logic
        if date < intervention_start:
            intervention_phase = 'Before'
            intervention_flag = 0
            case_trend = base_cases + np.random.randint(-5, 10)
        else:
            intervention_phase = 'After'
            intervention_flag = 1
            case_trend = base_cases * 0.7 + np.random.randint(-5, 5)

        confirmed_cases = max(
            int(case_trend * month_factor + np.random.normal(0, 5)), 0
        )

        # Deaths with realistic CFR range
        cfr_rate = np.random.uniform(0.08, 0.25)
        deaths = int(confirmed_cases * cfr_rate)

        data.append([
            date,
            state,
            confirmed_cases,
            deaths,
            intervention_phase,
            intervention_flag
        ])

# -----------------------------
# Create DataFrame
# -----------------------------
df = pd.DataFrame(data, columns=[
    'date',
    'state',
    'confirmed_cases',
    'deaths',
    'intervention_phase',
    'intervention_start'
])

# -----------------------------
# Derived Metrics
# -----------------------------
df['CFR'] = np.where(
    df['confirmed_cases'] > 0,
    (df['deaths'] / df['confirmed_cases']) * 100,
    0
)

df['rolling_avg_cases'] = (
    df.sort_values('date')
      .groupby('state')['confirmed_cases']
      .rolling(window=3, min_periods=1)
      .mean()
      .reset_index(level=0, drop=True)
)

# -----------------------------
# Save Dataset
# -----------------------------
df.to_csv('simulated_lassa_program_data_nigeria.csv', index=False)

df.head()

,date,state,confirmed_cases,deaths,intervention_phase,intervention_start,CFR,rolling_avg_cases
0,2019-01-01,Edo,25,2,Before,0,8.000000,25.000000
1,2019-02-01,Edo,41,6,Before,0,14.634146,33.000000
2,2019-03-01,Edo,40,3,Before,0,7.500000,35.333333
3,2019-04-01,Edo,21,4,Before,0,19.047619,34.000000
4,2019-05-01,Edo,24,2,Before,0,8.333333,28.333333


*Did the intervention actually change outcomes?*

In [2]:
impact_summary = (
    df.groupby(['intervention_phase'])
    .agg(
        avg_cases=('confirmed_cases', 'mean'),
        total_cases=('confirmed_cases', 'sum'),
        avg_deaths=('deaths', 'mean'),
        avg_cfr=('CFR', 'mean')
    )
    .reset_index()
)


impact_summary

,intervention_phase,avg_cases,total_cases,avg_deaths,avg_cfr
0,After,17.939815,3875,2.416667,12.677656
1,Before,28.687500,4131,4.416667,15.064677


*What is the Percentage Change?*

In [4]:
pre = impact_summary[impact_summary['intervention_phase'] == 'Before']
post = impact_summary[impact_summary['intervention_phase'] == 'After']

pct_change_cases = (
    (post['avg_cases'].values[0] - pre['avg_cases'].values[0])
    / pre['avg_cases'].values[0]
) * 100

pct_change_cases

np.float64(-37.4646978132817)

*Statistical Confirmation*

In [5]:
from scipy.stats import ttest_ind

before_cases = df[df['intervention_phase'] == 'Before']['confirmed_cases']
after_cases = df[df['intervention_phase'] == 'After']['confirmed_cases']

t_stat, p_value = ttest_ind(before_cases, after_cases, equal_var=False)

t_stat, p_value

(np.float64(9.951674461887134), np.float64(6.636372131985209e-20))

*Core Impact KPIs*

In [9]:
import matplotlib.pyplot as plt

plt.figure(figsize=(12,6))

for state in df ['state'].unique():
    state_df = df[df['state'] == state]
    plt.plot(state_df['date'], state_df['confirmed_cases'], alpha=0.3)

monthly_avg = df.groupby('date')['confirmed_cases'].mean().reset_index()
plt.plot(monthly_avg['date'], monthly_avg['confirmed_cases'],
        color='black', linewidth=3, label='National Average')


plt.axvline(pd.Timestamp('2021-01-01'), color='red', linestyle='--', label='Intervention Start')

plt.title('Confirmed Lassa Fever Cases Before and After Intervention')
plt.xlabel('Date')
plt.ylabel('Confirmed Cases')
plt.legend()
plt.tight_layout()
plt.savefig('intervention_impact_trend.png', dpi=300)
plt.close()

In [10]:
impact_summary = (
    df.groupby('intervention_phase')['confirmed_cases']
    .mean()
    .reset_index()
)

plt.figure(figsize=(6,4))
plt.bar(impact_summary['intervention_phase'], impact_summary['confirmed_cases'], color=['steelblue','seagreen'])
plt.title('Average Monthly Cases: Pre vs Post Intervention')
plt.ylabel('Confirmed Cases')
plt.tight_layout()
plt.savefig('pre_post_comparison.png', dpi=300)
plt.close()

In [12]:
cfr_trend = df.groupby('date')['CFR'].mean().reset_index()

plt.figure(figsize=(10,5))
plt.plot(cfr_trend['date'], cfr_trend['CFR'], color='purple')
plt.axvline(pd.Timestamp('2021-01-01'), color='red', linestyle='--')
plt.title('Case Fatality Rate (CFR) Over Time')
plt.ylabel('CFR (%)')
plt.xlabel('Date')
plt.tight_layout()
plt.savefig('cfr_trend_intervention.png', dpi=300)
plt.close()